# dynamic_soxx_soxl_cash_allocation_wf_v6

目标：验证 SOXX + SOXL + cash 动态配置在 open-to-open 口径下，加入收盘后可知的 same-day 15m 聚合特征后，是否能达到 SOXX +20pct 且最大回撤不低于 -10%。

本卡区分 outer-test capacity near-miss 和 nested threshold calibration，避免把阈值分布校准误当成可实盘模型。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while ROOT.name != 'workspace' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import qlib
from qlib.data import D
from qlib.workflow import R
from xgboost import XGBRegressor, XGBClassifier

from src.workflow.dynamic_soxx_soxl_cash_allocation_wf_v6 import (
    run_dynamic_soxx_soxl_cash_allocation_wf_v6_card,
)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
summary = run_dynamic_soxx_soxl_cash_allocation_wf_v6_card(root=ROOT)
summary

[86851:MainThread](2026-07-07 14:36:23,552) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86851:MainThread](2026-07-07 14:36:23,554) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86851:MainThread](2026-07-07 14:36:23,554) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/dynamic_soxx_soxl_cash_allocation_wf_v1')}


[86851:MainThread](2026-07-07 14:36:23,605) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86851:MainThread](2026-07-07 14:36:23,606) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86851:MainThread](2026-07-07 14:36:23,606) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[86851:MainThread](2026-07-07 14:36:29,012) INFO - qlib.workflow - [exp.py:258] - Experiment 612469460864566030 starts running ...


[86851:MainThread](2026-07-07 14:36:29,126) INFO - qlib.workflow - [recorder.py:345] - Recorder a8f05e338d7d4ed8a558ec737f5bfd24 starts running under Experiment 612469460864566030 ...


[86851:MainThread](2026-07-07 14:36:29,318) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'experiment': 'dynamic_soxx_soxl_cash_allocation_wf_v6',
 'card_type': 'dynamic_soxx_soxl_cash_same_day_15m_xgb_threshold_calibration_card',
 'promotion_decision': 'not_promoted_same_day_15m_capacity_and_nested_calibration_failed',
 'common_data_end': '2026-07-02',
 'source_experiment': 'dynamic_soxx_soxl_cash_allocation_wf_v1',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/dynamic_soxx_soxl_cash_allocation_wf_v1',
 'intraday_15m_provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min',
 'benchmark_asset': 'SOXX',
 'benchmark_basis': 'open_to_open',
 'allocation_assets': ['SOXX', 'SOXL', 'cash'],
 'execution_timing': 'same_day_15m_close_t_signal_next_open_to_following_open',
 'goal': {'excess_vs_soxx_floor_pct': 20.0, 'max_drawdown_floor_pct': -10.0},
 'benchmark': {'asset': 'SOXX',
  'return_pct': 413.65653034793365,
  'max_drawdown_pct': -43.091895274979755,
  'start': '2023-01-03',
  'end': '2026-06-30',
  'rows': 875},
 'feature_audit': {'daily_provid

In [3]:
import pandas as pd
capacity = pd.read_csv(ROOT / 'research/dynamic_soxx_soxl_cash_allocation_wf_v6/capacity_grid.csv')
nested = pd.read_csv(ROOT / 'research/dynamic_soxx_soxl_cash_allocation_wf_v6/nested_walk_forward.csv')
display(capacity.sort_values(['passes_target', 'excess_vs_soxx_pct', 'max_drawdown_pct'], ascending=[False, False, False]).head(10))
display(nested)

,qret,qtail,pupthr,fallback,soxl_w,return_pct,max_drawdown_pct,excess_vs_soxx_pct,passes_excess_gate,passes_drawdown_gate,passes_target,avg_soxx_weight,avg_soxl_weight,active_share,mid_share,selection_uses_outer_test,notes
131,0.44,0.96,0.58,0.28,0.95,449.701598,-10.396466,36.045068,True,False,False,0.130560,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
275,0.45,0.96,0.58,0.28,0.95,449.701598,-10.396466,36.045068,True,False,False,0.130560,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
419,0.46,0.96,0.58,0.28,0.95,449.701598,-10.396466,36.045068,True,False,False,0.130560,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
563,0.47,0.96,0.58,0.28,0.95,449.701598,-10.396466,36.045068,True,False,False,0.130560,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
128,0.44,0.96,0.58,0.26,0.95,441.725680,-10.318710,28.069150,True,False,False,0.121234,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
272,0.45,0.96,0.58,0.26,0.95,441.725680,-10.318710,28.069150,True,False,False,0.121234,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
416,0.46,0.96,0.58,0.26,0.95,441.725680,-10.318710,28.069150,True,False,False,0.121234,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
560,0.47,0.96,0.58,0.26,0.95,441.725680,-10.318710,28.069150,True,False,False,0.121234,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
125,0.44,0.96,0.58,0.25,0.95,437.764345,-10.279835,24.107815,True,False,False,0.116571,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...
269,0.45,0.96,0.58,0.25,0.95,437.764345,-10.279835,24.107815,True,False,False,0.116571,0.069486,0.073143,0.466286,True,outer_test_distribution_capacity_diagnostic_no...


,outer_year,qret,qtail,pupthr,fallback,soxl_w,validation_return_pct,validation_max_drawdown_pct,test_return_pct,test_max_drawdown_pct,test_avg_soxx_weight,test_avg_soxl_weight,test_active_share,test_mid_share,ret_threshold,tail_threshold,selection_uses_outer_test,notes
0,2023,0.47,0.96,0.585,0.24,0.9,222.212151,-48.917838,31.082741,-11.337869,0.088320,0.082800,0.092000,0.368000,0.002612,0.250628,False,nested_threshold_calibration_uses_prior_valida...
1,2024,0.47,0.96,0.585,0.24,0.9,331.265100,-48.481954,32.187485,-5.888504,0.138095,0.053571,0.059524,0.575397,0.002537,0.257197,False,nested_threshold_calibration_uses_prior_valida...
2,2025,0.44,0.95,0.585,0.24,0.9,144.942688,-48.481954,67.101870,-7.163515,0.131520,0.032400,0.036000,0.548000,0.001077,0.259727,False,nested_threshold_calibration_uses_prior_valida...
3,2026,0.44,0.93,0.585,0.24,0.9,155.308736,-48.542496,16.044131,-2.542924,0.076098,0.014634,0.016260,0.317073,0.001066,0.255548,False,nested_threshold_calibration_uses_prior_valida...


In [4]:
print((ROOT / 'research/dynamic_soxx_soxl_cash_allocation_wf_v6/conclusion.md').read_text(encoding='utf-8'))

# 中文结论

- 做了什么：新增 v6 same-day 15m + XGB 阈值校准实验；same-day 15m 指收盘后已经完整可知的当日 15 分钟聚合，用于 next-open 信号，不用于盘前信号。
- 容量结果：capacity pass count=0；收益达标侧 best return=433.8208%，maxDD=-10.2410%；回撤安全侧 best return=410.7294%，maxDD=-9.9279%。
- nested 结果：rolling validation window=4 年，stitched return=236.0008%，maxDD=-11.3379%，excess=-177.6557%。
- 为什么不晋升：正式 replay 下 capacity 已经是 near-miss；nested 阈值迁移后收益和回撤更不能同时过门槛。
- 是否晋升：不晋升；decision=not_promoted_same_day_15m_capacity_and_nested_calibration_failed。
- 下一步：优化方向应集中在在线阈值校准、score 分布漂移修正和按年度/波动 regime 的校准器，而不是继续扩大同一组 XGB 特征。
- qlib recorder：a8f05e338d7d4ed8a558ec737f5bfd24
- 不构成买卖建议。

